Quantization is a way of making models smaller and faster by reducing the precision of their weights and activations. Instead of using 32-bit floating point numbers, we can use 8-bit or even 4-bit integers, which saves memory and speeds up inference.

This notebook demonstrates how to apply quantization on a small model such as DistilBERT. We'll compare memory usage and run a quick inference test before and after quantization.

In [5]:
# Install required libraries
!pip install -q transformers accelerate bitsandbytes

### Quantization of models

In [ ]:
# Import libraries
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_id = "facebook/opt-1.3b"

tokenizer = AutoTokenizer.from_pretrained(model_id)

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

We define a function for testing the models. We are working with Causal Language Modelling here.

In [ ]:
def prediction(generator):
  prompt = "Once upon a time in a futuristic city,"
  output = generator(prompt, max_length=50, do_sample=True, temperature=0.7)

  print("Generated text:")
  print(output[0]['generated_text'])

By default, our model will be loaded in Float 32

In [ ]:
# Load a small model without quantization
model_fp32 = AutoModelForCausalLM.from_pretrained(model_id)

# Build a pipeline for sentiment analysis
generator_fp32 = pipeline("text-generation", model=model_fp32, tokenizer=tokenizer)

# Quick test before quantization
print("Full precision prediction:")

prediction(generator_fp32)

pytorch_model.bin:   0%|          | 0.00/2.63G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=256) and `max_length`(=51) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Full precision prediction:
Generated text:
Once upon a time in a futuristic city, a new world has begun. A new race of people has risen out of a great rift in the spacetime continuum. They have no name, no religion, and their true identity is unknown to them. Only they know that they are a race of superintelligent beings who can change the very fabric of reality.

The race is searching for a new world to call their own. They are trying to find a home in the distant galaxy to colonize and build new cities and new civilizations. They have not yet discovered the secrets of the universe, so they must rely on many technologies that are beyond their grasp.

The first city builder that the race has discovered is a game that is called "The City Builder". This game is an advanced simulation game that allows the players to build their own cities with some of the most advanced technology known to man.

However, the game has a flaw that has allowed players to cheat. The game allows the player to c

In [ ]:
mem_fp32 = torch.cuda.memory_allocated()
print(f"GPU memory (FP32): {mem_fp32 / 1e6:.2f} MB")

# Delete FP32 model to free memory
del model_fp32
torch.cuda.empty_cache()

GPU memory (FP32): 5734.07 MB


Now we load the model with 8-bit quantization through BitsAndBytes backend.

In [ ]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(load_in_8bit=True)

model_int8 = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,   # quantization
    device_map="auto"    # automatically place model on GPU if available
)

# print("\n8-bit quantized prediction:")

# generator_int8 = pipeline("text-generation", model=model_int8, tokenizer=tokenizer)

# prediction(generator_int8)

In [ ]:
mem_int8 = torch.cuda.memory_allocated()
print(f"GPU memory (INT8): {mem_int8 / 1e6:.2f} MB")

del mem_int8
torch.cuda.empty_cache()

GPU memory (INT8): 7162.49 MB


Finally we load the model with 4-bit quantization.

In [ ]:
# Try 4-bit quantization (more aggressive)
bnb_config = BitsAndBytesConfig(load_in_4bit=True)

model_int4 = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,   # quantization
    device_map="auto"    # automatically place model on GPU if available
)

In [ ]:
print("\n4-bit quantized prediction:")

generator_int4 = pipeline("text-generation", model=model_int4, tokenizer=tokenizer)

prediction(generator_int4)

Device set to use cuda:0
Both `max_new_tokens` (=256) and `max_length`(=51) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



4-bit quantized prediction:
Generated text:
Once upon a time in a futuristic city, a powerful alien race, whose names cannot be spoken, came to earth. They came to take what they wanted, kill what they could, and destroy what they could not. But this time, they came to fight.

Our Heroes

The heroes of this tale are the people of the world, who were born to fight evil, and to defend it. They are heroes who do not know their names, and who do not care. They are the ones who save the world, and the ones who are saved.

The Villain

The villain of this tale is the leader of the alien race that came to earth, who is known as the “Master”. He is the one who created the alien race, and who has been given the title of the “Duke”. He is also the one who has been given the task of “saving” the world, and of “saving” the people who have been saved. He is the one who has come to kill the heroes who have been saved, and to kill the ones who have been saved. The villain is a man who is known to be

In [ ]:
mem_int4 = torch.cuda.memory_allocated()
print(f"GPU memory (INT4): {mem_int4 / 1e6:.2f} MB")

del model_int4
torch.cuda.empty_cache()

GPU memory (INT4): 2747.41 MB


## Quantized LoRA

In QLoRA:
- The base model is stored in 4-bit (or 8-bit) precision (saving tons of GPU memory).
- LoRA adapters are added in normal precision and trained.
- During forward pass, quantized weights are dequantized just-in-time for computation.

**Advantages**.
- Allows fine-tuning very large models (e.g., 65B parameters) on a single GPU.
- Hugely reduces memory usage: e.g., a 33B model that might need >100 GB can fit into ~24 GB GPU RAM.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# Step 1: Load base model in 4-bit precision
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(
    "openai-community/gpt2-large",
    quantization_config=bnb_config,
    device_map="auto"
)

# Step 2: Define LoRA config (tiny adapters)
config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,               # rank (low-rank dimension)
    lora_alpha=32,
    lora_dropout=0.1, # drop 1% of activations randomly so it doesn't overfit
    bias="none", # keep all biases frozen.
    target_modules=["attn.c_attn", "attn.c_proj"]
)

# Step 3: Prepare quantizied for training
model = prepare_model_for_kbit_training(model)

# Step 3: Wrap model with PEFT
model = get_peft_model(model, config)
model.print_trainable_parameters()

# continue as usual

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.25G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

trainable params: 2,211,840 || all params: 776,241,920 || trainable%: 0.2849


## Compare memory use and speed before vs. after LoRA tuning.

#### Memory use

For full fine-tuning every model weight requires gradient storage during training. So you store parameters + gradients.

In LoRA, only the LoRA matrices need gradients.

#### Speed

Full fine-tuning: full backpropagation over all parameters → slower training.

In LoRA, only a few extra multiplications for the adapter layers during backprop.



In [4]:
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

# Load tokenizer and model
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).cuda()

# Dummy input for benchmarking
input_text = "Hello, world! " * 5
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

# -------------------------
# Function to benchmark memory and speed

def benchmark_model(model, inputs, steps=10):
    torch.cuda.reset_peak_memory_stats() # Clears the previous GPU memory tracking.
    start_time = time.time()

    for _ in range(steps):
        outputs = model(**inputs, labels=inputs["input_ids"])  # forward pass
        loss = outputs.loss
        loss.backward()  # simulate training step, calculate gradients
        model.zero_grad() # resets gradients to zero for the next iteration

    end_time = time.time()
    peak_memory = torch.cuda.max_memory_allocated() / 1024**2  # MB
    avg_time = (end_time - start_time) / steps  # seconds per step
    return peak_memory, avg_time

# -------------------------
# Benchmark before LoRA

print("Benchmarking GPT-2 full fine-tuning...")
mem_before, speed_before = benchmark_model(model, inputs)
print(f"Peak Memory: {mem_before:.2f} MB, Avg Step Time: {speed_before:.4f}s")

# -------------------------
# Apply LoRA

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    target_modules=["attn.c_attn", "attn.c_proj"]
)
model = get_peft_model(model, lora_config)

# -------------------------
# Benchmark after LoRA

print("Benchmarking GPT-2 with LoRA...")
mem_after, speed_after = benchmark_model(model, inputs)
print(f"Peak Memory: {mem_after:.2f} MB, Avg Step Time: {speed_after:.4f}s")

# -------------------------
print("\nSummary:")
print(f"Memory reduction: {mem_before - mem_after:.2f} MB")
print(f"Speed improvement: {speed_before / speed_after:.2f}x faster")


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

... (2 outputs left out)

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Peak Memory: 1905.42 MB, Avg Step Time: 0.1211s
Benchmarking GPT-2 with LoRA...


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2174: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


Peak Memory: 1166.00 MB, Avg Step Time: 0.0349s

Summary:
Memory reduction: 739.42 MB
Speed improvement: 3.48x faster
